# 01 — Getting Started with the Sentinel Data Lake

**Goal:** authenticate, connect to the Log Analytics workspace holding `GigamonCcfMcpDemo_CL`, and run the first KQL query.

This notebook is the foundation every other notebook in this repo builds on.

## Setup

Authenticate with Azure CLI (`az login`) and load workspace coordinates from `.env`.

In [1]:
import os, datetime as dt
import pandas as pd
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.monitor.query import LogsQueryClient, LogsQueryStatus

load_dotenv()
WORKSPACE_ID = os.environ['WORKSPACE_ID']
HOURS = int(os.environ.get('TIMERANGE_HOURS', '24'))
client = LogsQueryClient(DefaultAzureCredential())
TIMESPAN = dt.timedelta(hours=HOURS)

def kql(q: str) -> pd.DataFrame:
    """Run a KQL query and return the first table as a DataFrame."""
    r = client.query_workspace(WORKSPACE_ID, q, timespan=TIMESPAN)
    if r.status != LogsQueryStatus.SUCCESS:
        raise RuntimeError(r.partial_error)
    t = r.tables[0]
    return pd.DataFrame(t.rows, columns=[c for c in t.columns])

/Users/mitchellgulledge/gigamon-sentinel-notebooks/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Sanity check — how much Gigamon data do we have?

In [2]:
df = kql('''
GigamonCcfMcpDemo_CL
| summarize Events=count(),
            FirstSeen=min(TimeGenerated),
            LastSeen=max(TimeGenerated),
            UniqueSources=dcount(src_ip),
            UniqueDestinations=dcount(dst_ip),
            UniqueApps=dcount(app_name)
''')
df

,Events,FirstSeen,LastSeen,UniqueSources,UniqueDestinations,UniqueApps
0,1500,2026-05-12 21:21:52.576000+00:00,2026-05-13 17:25:27.695000+00:00,5,5,22


## What protocols / apps are present?

In [3]:
kql('''
GigamonCcfMcpDemo_CL
| summarize Flows=count() by app_family, protocol
| top 20 by Flows
''')

,app_family,protocol,Flows
0,remote-admin,TCP,130
1,dns,UDP,126
2,dns,TCP,122
3,web,UDP,114
4,file-sharing,TCP,110
5,web,TCP,100
6,file-sharing,UDP,100
7,secure-shell,UDP,98
8,secure-shell,TCP,89
9,remote-admin,UDP,81


## Next steps

- `02` — Lateral movement graph
- `03` — JA3 fingerprint hunting
- `04` — Beacon periodicity
- `05` — App-mix dashboard